# Naive RAG from scratch
Load a PDF → chunk → embed → store in Chroma → retrieve → answer with Gemini.

Open in Google Colab, upload any PDF as `doc.pdf`, and add your Gemini key (free at https://aistudio.google.com/apikey).

In [ ]:
!pip install -q chromadb sentence-transformers pypdf langchain-text-splitters google-genai

## 1. Load

In [ ]:
from pypdf import PdfReader

text = "".join(p.extract_text() or "" for p in PdfReader("doc.pdf").pages)
print(len(text), "characters")

## 2. Chunk
Try `chunk_size` 200 and 1000 later and compare the answers.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)
print(len(chunks), "chunks")
print(chunks[0])

## 3. Embed and store

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(chunks, normalize_embeddings=True)
print(vectors.shape)  # (num_chunks, 384)

col = chromadb.Client().get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})
col.add(ids=[str(i) for i in range(len(chunks))], documents=chunks, embeddings=vectors.tolist())

## 4. Retrieve
Lower distance = more similar.

In [ ]:
question = "What is this document about?"
res = col.query(query_embeddings=model.encode([question], normalize_embeddings=True).tolist(), n_results=3)
for cid, doc, dist in zip(res["ids"][0], res["documents"][0], res["distances"][0]):
    print(f"[{cid}] distance={dist:.3f}\n{doc[:200]}\n")

## 5. Generate

In [ ]:
from google import genai

client = genai.Client(api_key="YOUR_GEMINI_KEY")
context = "\n\n".join(f"[{i}] {d}" for i, d in zip(res["ids"][0], res["documents"][0]))
prompt = f"""Answer using ONLY the context. Cite chunk ids in [brackets].
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}"""
print(client.models.generate_content(model="gemini-3.6-flash", contents=prompt).text)

## My observations
- chunk_size 200 vs 500 vs 1000:
- n_results 1 vs 3 vs 5:
- A question it got wrong and why: